# RAG

In [23]:
API = "http://export.arxiv.org/api/query"
THEME_KEYWORDS = ["exoplanet", "planet detection", "habitability", "TRAPPIST", "TESS"]
# THEME_KEYWORDS = ["exoplanet", "TRAPPIST"]
OUTPUT_FILE = "exoplanet_data.json"
MAX_DATA_SIZE = 2000

In [ ]:
import requests
import xml.etree.ElementTree as ET
import json
import os
import time

def get_exoplanet_data_xml(start=0, batch_size=100):
    search_terms = " OR ".join([f'all:{keyword}' for keyword in THEME_KEYWORDS])
    query = f"search_query=({search_terms})&start={start}&max_results={batch_size}"
    url = f"{API}?{query}"
    
    response = requests.get(url)
    response.raise_for_status()
    
    root = ET.fromstring(response.content)
    return root

def read_xml(xml_root: ET.Element, output):
    namespace = {"atom": "http://www.w3.org/2005/Atom"}
    rows_added = 0
    
    for entry in xml_root.findall('atom:entry', namespace):
        authors = entry.findall('atom:author', namespace)
        
        categories = entry.findall('atom:category', namespace)
        categories_list = [cat.get('term') for cat in categories] if categories else []
        
        title_elem = entry.find('atom:title', namespace)
        id_elem = entry.find('atom:id', namespace)
        published_elem = entry.find('atom:published', namespace)
        summary_elem = entry.find('atom:summary', namespace)
        
        if title_elem is None or id_elem is None or published_elem is None or summary_elem is None:
            continue
        
        author_names = []
        for author in authors:
            name_elem = author.find('atom:name', namespace)
            if name_elem is not None and name_elem.text:
                author_names.append(name_elem.text)
        
        row = {
            "title": title_elem.text.strip() if title_elem.text else "",
            "id": id_elem.text if id_elem.text else "",
            "published": published_elem.text if published_elem.text else "",
            "summary": summary_elem.text.strip() if summary_elem.text else "",
            "authors": author_names,
            "categories": categories_list
        }
        output.append(row)
        rows_added += 1
        
    return rows_added

def fetch_exoplanet_data(output_file=OUTPUT_FILE):
    batch_size = 100
    total_rows = 0
    data = []
    
    print("Start fetching exoplanet data...")
    
    while total_rows < MAX_DATA_SIZE:
        print(f"Fetching data from {total_rows} to {total_rows + batch_size}")
        
        try:
            xml_root = get_exoplanet_data_xml(total_rows, batch_size)
            rows_added = read_xml(xml_root, data)
            
            if rows_added == 0:
                print("No more data found, exiting...")
                break
                
            total_rows += rows_added
            print(f"Added {rows_added} rows to {output_file}, total rows: {total_rows}")
            time.sleep(1)
            
        except Exception as e:
            print(f"Error fetching data: {e}")
            break
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    print(f"Data saved to {output_file}")
    
    return data

articles = fetch_exoplanet_data()

Start fetching exoplanet data...
Fetching data from 0 to 100
Added 100 rows to exoplanet_data.json, total rows: 100
Fetching data from 100 to 200
Added 100 rows to exoplanet_data.json, total rows: 200
Fetching data from 200 to 300
Added 100 rows to exoplanet_data.json, total rows: 300
Fetching data from 300 to 400
Added 100 rows to exoplanet_data.json, total rows: 400
Fetching data from 400 to 500
Added 100 rows to exoplanet_data.json, total rows: 500
Fetching data from 500 to 600
Added 100 rows to exoplanet_data.json, total rows: 600
Fetching data from 600 to 700
Added 100 rows to exoplanet_data.json, total rows: 700
Fetching data from 700 to 800
No more data found, exiting...
Data saved to exoplanet_data.json


In [25]:
def load_data_from_json(file_path=OUTPUT_FILE):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

articles = load_data_from_json()

In [26]:
print(f"Всего загружено публикаций: {len(articles)}")
print(f"Первые 3 публикации:")

for i, article in enumerate(articles[:3]):
    print(f"\n{i+1}. {article['title']}")
    print(f"   Авторы: {', '.join(article['authors'][:3])}{'...' if len(article['authors']) > 3 else ''}")
    print(f"   Категории: {', '.join(article['categories'])}")
    print(f"   Опубликовано: {article['published'][:10]}")
    print(f"   Аннотация: {article['summary'][:200]}...")


Всего загружено публикаций: 700
Первые 3 публикации:

1. TESS Habitable Zone Star Catalog
   Авторы: L. Kaltenegger, J. Pepper, K. Stassun...
   Категории: astro-ph.EP
   Опубликовано: 2019-03-27
   Аннотация: We present the Transiting Exoplanet Survey Satellite (TESS) Habitable Zone
Stars Catalog, a list of 1822 nearby stars with a TESS magnitude brighter than
T = 12 and reliable distances from Gaia DR2, a...

2. Around which stars can TESS detect Earth-like planets? The Revised TESS
  Habitable Zone Catalog
   Авторы: L. Kaltenegger, J. Pepper, P. M. Christodoulou...
   Категории: astro-ph.EP, astro-ph.IM, astro-ph.SR
   Опубликовано: 2021-01-19
   Аннотация: In the search for life in the cosmos, NASA's Transiting Exoplanet Survey
Satellite (TESS) mission has already monitored about 74% of the sky for
transiting extrasolar planets, including potentially ha...

3. Analyzing the Habitable Zones of Circumbinary Planets Using Machine
  Learning
   Авторы: Zhihui Kong, Jonathan H. Jiang, 

In [27]:
from collections import Counter

all_categories = []
for article in articles:
    all_categories.extend(article['categories'])

# - atstro-ph.EP - Earth and planetary physics
# - astro-ph.SR - Sollar ans Stellar astrophysics
# - astro-ph.IM - Instrumentation and Methods for Astrophysics
# - astro-ph.GA - Astrophysics of Galaxies

category_counts = Counter(all_categories)
print("Топ-10 категорий публикаций:")
for category, count in category_counts.most_common(10):
    print(f"  {category}: {count} публикаций")


Топ-10 категорий публикаций:
  astro-ph.EP: 668 публикаций
  astro-ph.SR: 173 публикаций
  astro-ph.IM: 159 публикаций
  astro-ph.GA: 20 публикаций
  astro-ph: 16 публикаций
  physics.ao-ph: 14 публикаций
  cs.LG: 11 публикаций
  physics.space-ph: 5 публикаций
  physics.geo-ph: 4 публикаций
  physics.pop-ph: 3 публикаций


## Разбиение текста на чанки

Разобьем статьт на более мелкие части - чанки


In [28]:
import re

CHUNK_SIZE = 500
OVERLAP_SIZE = 50

def clean_text(text):
    # лишние пробелы и переносы строк
    text = re.sub(r'\s+', ' ', text.strip())
    # специальные символы, оставляя только буквы, цифры и пунктуацию
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)]', ' ', text)
    return text

def split_into_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap_size: int = OVERLAP_SIZE):
    words = text.split()
    
    if len(words) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk_words = words[start:end]
        chunk_text = ' '.join(chunk_words)
        chunks.append(chunk_text)
        
        if end >= len(words):
            break
            
        start = end - overlap_size
        
        if start >= end:
            break
    
    return chunks

def create_chunks_from_articles(articles):
    all_chunks = []
    
    for article_idx, article in enumerate(articles):
        title = article.get('title', '')
        summary = article.get('summary', '')

        full_text = f"{title}. {summary}"
        full_text = clean_text(full_text)

        chunks = split_into_chunks(full_text)
        
        for chunk_idx, chunk_text in enumerate(chunks):
            chunk_data = {
                'chunk_id': f"{article_idx}_{chunk_idx}",
                'article_id': article_idx,
                'chunk_index': chunk_idx,
                'text': chunk_text,
                'article_title': title,
                'article_authors': article.get('authors', []),
                'article_categories': article.get('categories', []),
                'article_published': article.get('published', ''),
                'word_count': len(chunk_text.split())
            }
            all_chunks.append(chunk_data)
    
    return all_chunks

chunks = create_chunks_from_articles(articles)

print(f"Создано чанков: {len(chunks)}")
print(f"В среднем чанков на статью: {len(chunks) / len(articles):.2f}")


Создано чанков: 700
В среднем чанков на статью: 1.00


In [29]:
from collections import Counter

print(f"Общее количество чанков: {len(chunks)}")

chunk_sizes = [chunk['word_count'] for chunk in chunks]
print(f"Размер чанков:")
print(f"  - Минимальный: {min(chunk_sizes)} слов")
print(f"  - Максимальный: {max(chunk_sizes)} слов") 
print(f"  - Средний: {sum(chunk_sizes)/len(chunk_sizes):.1f} слов")

articles_chunk_count = Counter([chunk['article_id'] for chunk in chunks])
chunk_counts = list(articles_chunk_count.values())

print(f"\nЧанков на статью:")
print(f"  - Минимум: {min(chunk_counts)} чанков")
print(f"  - Максимум: {max(chunk_counts)} чанков")
print(f"  - Среднее: {sum(chunk_counts)/len(chunk_counts):.1f} чанков")


Общее количество чанков: 700
Размер чанков:
  - Минимальный: 56 слов
  - Максимальный: 339 слов
  - Средний: 232.1 слов

Чанков на статью:
  - Минимум: 1 чанков
  - Максимум: 1 чанков
  - Среднее: 1.0 чанков


In [30]:
import json

CHUNKS_FILE = "exoplanet_chunks.json"
with open(CHUNKS_FILE, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Чанки сохранены в файл: {CHUNKS_FILE}")
# Создаем также простой список текстов для TF-IDF анализа
chunk_texts = [chunk['text'] for chunk in chunks]
print(f"Подготовлен список из {len(chunk_texts)} текстов чанков для анализа")


Чанки сохранены в файл: exoplanet_chunks.json
Подготовлен список из 700 текстов чанков для анализа



**Размер чанков:** Все чанки получились меньше 500 слов (максимум 339, средний 232 слова). 

Можно было бы уменьшить размер чанка, но кажется, что средний размер в 232 слова и так является оптимальным



## Векторизация текста и построение FAISS индекса

Для RAG-поиска преобразуем тексты в векторные представления (эмбеддинги) и создадим индекс для быстрого поиска.


In [31]:
from sentence_transformers import SentenceTransformer

print("Загрузка модели эмбеддингов...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Модель загружена. Размерность эмбеддингов: {model.get_sentence_embedding_dimension()}")


Загрузка модели эмбеддингов...
Модель загружена. Размерность эмбеддингов: 384


In [34]:
import numpy as np

def create_embeddings(chunk_texts):
    batch_size = 10
    embeddings = []

    for i in range(0, len(chunk_texts), batch_size):
        batch = chunk_texts[i:i+batch_size]
        batch_embeddings = model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
        embeddings.append(batch_embeddings)
        
        if (i // batch_size + 1) % 10 == 0:
            print(f"Обработано {min(i + batch_size, len(chunk_texts))} из {len(chunk_texts)} чанков")

    # Объединение в одну матрицу
    return np.vstack(embeddings)

In [35]:
chunk_texts = [chunk['text'] for chunk in chunks]
embeddings = create_embeddings(chunk_texts)
print(f"Форма матрицы эмбеддингов: {embeddings.shape}")
print(f"Размерность каждого эмбеддинга: {embeddings.shape[1]}")


Обработано 100 из 700 чанков
Обработано 200 из 700 чанков
Обработано 300 из 700 чанков
Обработано 400 из 700 чанков
Обработано 500 из 700 чанков
Обработано 600 из 700 чанков
Обработано 700 из 700 чанков
Форма матрицы эмбеддингов: (700, 384)
Размерность каждого эмбеддинга: 384


Поистройка индекса

In [36]:
import faiss

faiss.normalize_L2(embeddings)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype('float32'))

print(f"Индекс построен!")
print(f"Количество векторов в индексе: {index.ntotal}")
print(f"Размерность векторов: {index.d}")


Индекс построен!
Количество векторов в индексе: 700
Размерность векторов: 384


In [37]:
import numpy as np
import pickle

FIASS_INDEX_PATH = "faiss_index.bin"
METADATA_PATH = "chunks_metadata.pkl"

def save_faiss_index(index, index_path=FIASS_INDEX_PATH, metadata_path=METADATA_PATH):
    print("Сохранение FAISS индекса и метаданных...")

    faiss.write_index(index, index_path)
    print(f"FAISS индекс сохранен в: {index_path}")

    with open(metadata_path, 'wb') as f:
        pickle.dump(chunks, f)
    print(f"Метаданные чанков сохранены в: {metadata_path}")

def load_faiss_index(index_path=FIASS_INDEX_PATH, metadata_path=METADATA_PATH):
    loaded_index = faiss.read_index(index_path)
    with open(metadata_path, 'rb') as f:
        loaded_chunks = pickle.load(f)
    
    return loaded_index, loaded_chunks


In [38]:
save_faiss_index(index)

Сохранение FAISS индекса и метаданных...
FAISS индекс сохранен в: faiss_index.bin
Метаданные чанков сохранены в: chunks_metadata.pkl


Теперь напишем саму функцию для семантического поиска

In [39]:
def search_similar_chunks(query_text, top_k=5):
    query_embedding = model.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        chunk = chunks[idx]
        results.append({
            'chunk_id': chunk['chunk_id'],
            'similarity': float(dist),
            'text': chunk['text'],
            'article_title': chunk['article_title'],
            'article_authors': chunk['article_authors'],
            'article_categories': chunk['article_categories'],
            'word_count': chunk['word_count']
        })
    
    return results


In [40]:

test_queries = [
    "What are habitable exoplanets?",
    "How does TESS detect planets?",
    "What is the TRAPPIST system?",
]

for query in test_queries:
    print(f"Запрос: '{query}'")
    results = search_similar_chunks(query, top_k=1)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Сходство: {result['similarity']:.4f}")
        print(f"   Статья: {result['article_title']}")
        print(f"   Авторы: {', '.join(result['article_authors'][:2])}")
        print(f"   Текст: {result['text'][:200]}...")

    print("\n" + "=" * 80 + "\n")

Запрос: 'What are habitable exoplanets?'

1. Сходство: 0.7225
   Статья: Characterizing Exoplanet Habitability
   Авторы: Ravi kumar Kopparapu, Eric T. Wolf
   Текст: Characterizing Exoplanet Habitability. Habitability is a measure of an environment s potential to support life, and a habitable exoplanet supports liquid water on its surface. However, a planet s succ...


Запрос: 'How does TESS detect planets?'

1. Сходство: 0.7130
   Статья: The Transiting Exoplanet Survey Satellite
   Авторы: Joshua N. Winn
   Текст: The Transiting Exoplanet Survey Satellite. A transiting planet invites us to measure its size, mass, orbital parameters, atmospheric composition, and other characteristics. But the invitation can only...


Запрос: 'What is the TRAPPIST system?'

1. Сходство: 0.3703
   Статья: Planet-Planet Tides in the TRAPPIST-1 System
   Авторы: Jason T. Wright
   Текст: Planet-Planet Tides in the TRAPPIST-1 System. The star TRAPPIST-1 hosts a system of seven transiting, terrestrial exop

## Интеграция с LLM для генерации ответов

Теперь настроим интеграцию с LLM для генерации ответов. Для работы я воспользуюсь FLAN-5


In [83]:
def create_rag_prompt(query: str, context_chunks, max_context_length: int = 2000) -> str:
    context_parts = []
    current_length = 0
    
    for i, chunk in enumerate(context_chunks, 1):
        chunk_text = f"[Source {i}]: {chunk['text']}"
        if current_length + len(chunk_text) > max_context_length:
            break
        context_parts.append(chunk_text)
        current_length += len(chunk_text)
    
    context = "\n\n".join(context_parts)
    prompt = f"""Using provided context from science articles about exoplanets and astronomy, answer the user's question.

Context:
{context}

Question: {query}

Instructions:
- Answer the question based only on the provided context
- If there is not enough information in the context, say so
- Be precise and use scientific terminology
- If specific studies or data are mentioned, indicate this

Answer:"""
    
    return prompt

example_chunks = [
    {'text': 'TESS mission has discovered many exoplanets using the transit method...'},
    {'text': 'Habitable exoplanets are planets that can support liquid water...'}
]
example_prompt = create_rag_prompt("What is TESS?", example_chunks)
print(example_prompt)


Using provided context from science articles about exoplanets and astronomy, answer the user's question.

Context:
[Source 1]: TESS mission has discovered many exoplanets using the transit method...

[Source 2]: Habitable exoplanets are planets that can support liquid water...

Question: What is TESS?

Instructions:
- Answer the question based only on the provided context
- If there is not enough information in the context, say so
- Be precise and use scientific terminology
- If specific studies or data are mentioned, indicate this

Answer:


In [42]:

from transformers import T5ForConditionalGeneration, T5Tokenizer
import sentencepiece
import torch

print("Загрузка модели FLAN-T5...")
model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
llm_model = T5ForConditionalGeneration.from_pretrained(model_name).to("cpu")

print("Модель FLAN-T5 загружена!")


Загрузка модели FLAN-T5...
Модель FLAN-T5 загружена!


In [69]:
def generate_answer_with_flan_t5(prompt: str, max_length: int = 512) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cpu")
    
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            do_sample=False,
            temperature=0.7
        )
    
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

test_prompt = "Question: What is an exoplanet? Answer:"
test_answer = generate_answer_with_flan_t5(test_prompt)
print(f"Тестовый ответ: {test_answer}")

Тестовый ответ: asteroid


In [52]:
def rag_query(query: str, top_k: int = 3, max_context_length: int = 2000):
    relevant_chunks = search_similar_chunks(query, top_k=top_k)
    prompt = create_rag_prompt(query, relevant_chunks, max_context_length=max_context_length)
    answer = generate_answer_with_flan_t5(prompt)
    
    result = {
        'query': query,
        'answer': answer,
        'sources': [
            {
                'title': chunk['article_title'],
                'authors': chunk['article_authors'],
                'similarity': chunk['similarity']
            }
            for chunk in relevant_chunks
        ],
        'chunks_used': relevant_chunks
    }
    
    return result


In [81]:
def ask_question(question: str, top_k: int = 5):
    result = rag_query(question, top_k=top_k)
    
    print(f"ВОПРОС: {result['query']}")
    print('='*80)
    print(f"\nОТВЕТ:\n{result['answer']}")
    print(f"\nИсточники ({len(result['sources'])}):")
    for i, source in enumerate(result['sources'], 1):
        print(f"  {i}. {source['title']}")
        print(f"     Сходство: {source['similarity']:.4f}")
    
    return result



In [84]:
result = ask_question("What are exoplanets?")

ВОПРОС: What are exoplanets?

ОТВЕТ:
Be precise and use scientific terminology

Источники (5):
  1. Resource Letter Exo-1: Exoplanets
     Сходство: 0.7684
  2. Exoplanet Predictions Based on the Generalised Titius-Bode Relation
     Сходство: 0.6944
  3. The search for exomoons and the characterization of exoplanet
  atmospheres
     Сходство: 0.6646
  4. Wide-Orbit Exoplanet Demographics
     Сходство: 0.6598
  5. Impact of Tides on the Potential for Exoplanets to Host Exomoons
     Сходство: 0.6502


## Промпт-инжиниринг

В этом разделе мы:
1. Улучшим промпты для лучшего извлечения ключевых идей из текста
2. Добавим обработку опечаток в пользовательских запросах
3. Протестируем систему на различных примерах
4. Покажем, что RAG действительно улучшает качество ответов LLM


In [182]:
from rapidfuzz import fuzz, process
from rapidfuzz.distance import Levenshtein

def fix_spelling(query: str, vocabulary=None):
    if vocabulary is None:
        all_words = set()
        for text in chunk_texts:
            words = text.lower().split()
            all_words.update(words)
        vocabulary = list(all_words)
    
    query_words = query.split()
    corrected_words = []
    
    for word in query_words:
        if (word in vocabulary or word.lower() in vocabulary):
            corrected_words.append(word)
            continue

        if len(word) <= 2 or not word.isalnum():
            corrected_words.append(word)
            continue
        
        best_match = process.extractOne(word.lower(), vocabulary, scorer=Levenshtein.distance)
        
        if best_match:
            print(best_match)
            corrected_word = best_match[0]
            if word[0].isupper():
                corrected_word = corrected_word.capitalize()
            corrected_words.append(corrected_word)
        else:
            corrected_words.append(word)
    
    return ' '.join(corrected_words)


In [175]:
process.extractOne("expplant", ["exoplanet", "planet", "exoplanets", "exploit"], scorer=Levenshtein.distance)

('exoplanet', 2, 0)

In [184]:
test_queries = [
    "What is expplanet?",
    "What is the TRAPIST system?",
]

print("Тестирование обработки опечаток:")
for query in test_queries:
    corrected = fix_spelling(query)
    if corrected != query:
        print(f"  '{query}' -> '{corrected}'")
    else:
        print(f"  '{query}' -> (без изменений)")

Тестирование обработки опечаток:
  'What is expplanet?' -> (без изменений)
('trappist', 1, 126)
  'What is the TRAPIST system?' -> 'What is the Trappist system?'


Почему то исправление ошибок не работает нормально.

In [197]:
def create_improved_rag_prompt(query: str, context_chunks, prompt_type="detailed", max_context_length: int = 2000):
    context_parts = []
    current_length = 0
    
    for i, chunk in enumerate(context_chunks, 1):
        chunk_text = f"[Source {i}]: {chunk['text']}"
        if current_length + len(chunk_text) > max_context_length:
            break
        context_parts.append(chunk_text)
        current_length += len(chunk_text)
    
    context = "\n\n".join(context_parts)
    
    # Различные типы промптов
    prompts = {
        "detailed": f"""Based on the provided scientific context about exoplanets and astronomy, provide a comprehensive answer to the question.

Context:
{context}

Question: {query}

Instructions:
- Extract and summarize the key information relevant to the question
- Identify main findings, methods, or conclusions mentioned in the context
- Use precise scientific terminology
- If specific data, numbers, or studies are mentioned, include them in your answer
- If the context doesn't contain enough information, clearly state what is missing

Answer:""",

        "key_points": f"""Extract key points and main ideas from the scientific context to answer the question.

Context:
{context}

Question: {query}

Task:
1. Identify the most relevant information from the context
2. Extract key findings, methods, or results
3. Summarize the main ideas in a clear and concise way
4. Highlight any specific data, numbers, or important details

Answer:""",

        "structured": f"""Answer the question using the provided scientific context. Structure your response clearly.

Context:
{context}

Question: {query}

Response format:
- Main answer: [direct answer to the question]
- Key findings: [important discoveries or results mentioned]
- Methods/Approach: [how the research was conducted, if mentioned]
- Limitations: [what information is missing or unclear]

Answer:""",

        "scientific": f"""As a scientific expert, analyze the provided research context and answer the question with scientific rigor.

Context:
{context}

Question: {query}

Analysis requirements:
- Extract quantitative data (numbers, measurements, statistics) if present
- Identify research methods and observational techniques mentioned
- Note any scientific models, theories, or frameworks discussed
- Highlight empirical findings and their significance
- Use appropriate scientific terminology

Scientific answer:""",

        "concise": f"""Provide a concise, factual answer based on the scientific context.

Context:
{context}

Question: {query}

Answer directly and factually, focusing on the most relevant information from the context:"""
    }
    
    return prompts.get(prompt_type, prompts["detailed"])


In [ ]:
def improved_rag_query(query: str, top_k: int = 3, prompt_type: str = "detailed", 
                       correct_spelling: bool = True, max_context_length: int = 2000):
    original_query = query
    if correct_spelling:
        query = fix_spelling(query)
        if query != original_query:
            print(f"Исправлен запрос: '{original_query}' -> '{query}'")

    relevant_chunks = search_similar_chunks(query, top_k=top_k)
    
    if not relevant_chunks:
        return {
            'query': original_query,
            'corrected_query': query if query != original_query else None,
            'answer': 'Не найдено релевантных документов для ответа на вопрос.',
            'sources': [],
            'chunks_used': []
        }
    
    prompt = create_improved_rag_prompt(query, relevant_chunks, prompt_type=prompt_type, 
                                       max_context_length=max_context_length)
    
    answer = generate_answer_with_flan_t5(prompt)
    
    result = {
        'query': original_query,
        'corrected_query': query if query != original_query else None,
        'answer': answer,
        'prompt_type': prompt_type,
        'sources': [
            {
                'title': chunk['article_title'],
                'authors': chunk['article_authors'],
                'similarity': chunk['similarity']
            }
            for chunk in relevant_chunks
        ],
        'chunks_used': relevant_chunks
    }
    
    return result


In [198]:
def compare_with_without_rag(query: str, top_k: int = 3, prompt_type: str = "detailed"):
    no_rag_prompt = f"Question: {query}\n\nAnswer:"
    answer_without_rag = generate_answer_with_flan_t5(no_rag_prompt)
    
    result_with_rag = improved_rag_query(query, top_k=top_k, prompt_type=prompt_type)
    answer_with_rag = result_with_rag['answer']
    
    return {
        'query': query,
        'answer_without_rag': answer_without_rag,
        'answer_with_rag': answer_with_rag,
        'sources': result_with_rag['sources']
    }

In [199]:
test_questions = [
    {
        'query': 'What are habitable exoplanets?',
        'description': 'Базовый вопрос о концепции',
        'prompt_type': 'detailed'
    },
    {
        'query': 'How does TESS detect planets?',
        'description': 'Вопрос о методе обнаружения',
        'prompt_type': 'structured'
    },
    {
        'query': 'What is the TRAPPIST system?',
        'description': 'Вопрос о конкретной системе',
        'prompt_type': 'key_points'
    },
    {
        'query': 'What methods are used to detect exoplanets?',
        'description': 'Вопрос о методах исследования',
        'prompt_type': 'scientific'
    },
    {
        'query': 'What are the main characteristics of exoplanets in habitable zones?',
        'description': 'Сложный вопрос с несколькими аспектами',
        'prompt_type': 'detailed'
    },
    {
        'query': 'How many exoplanets has Kepler discovered?',
        'description': 'Вопрос с конкретными данными',
        'prompt_type': 'scientific'
    },
    {
        'query': 'What is transit photometry?',
        'description': 'Вопрос о техническом методе',
        'prompt_type': 'concise'
    }
]



In [200]:
# Тестирование каждого вопроса с демонстрацией улучшения качества

for i, test_case in enumerate(test_questions, 1):
    query = test_case['query']
    prompt_type = test_case['prompt_type']
    
    print(f"\n{'='*80}")
    print(f"ТЕСТ {i}/{len(test_questions)}")
    print(f"Вопрос: {query}")
    print(f"Тип промпта: {prompt_type}")
    print('='*80)
    
    # Сравнение с RAG и без RAG
    comparison = compare_with_without_rag(query, top_k=10, prompt_type=prompt_type)
    
    print(f"\nОтвет без RAG:")
    print(f"{comparison['answer_without_rag']}")
    
    print(f"\nОтвет с RAG:")
    print(f"{comparison['answer_with_rag']}")
    
    # print(f"\nИСТОЧНИКИ ({len(comparison['sources'])}):")
    # for j, source in enumerate(comparison['sources'], 1):
    #     print(f"  {j}. {source['title']} (сходство: {source['similarity']:.4f})")
    
    print("\n" + "-"*80)



ТЕСТ 1/7
Вопрос: What are habitable exoplanets?
Тип промпта: detailed

Ответ без RAG:
asteroid

Ответ с RAG:
A measure of an environment s potential to support life

--------------------------------------------------------------------------------

ТЕСТ 2/7
Вопрос: How does TESS detect planets?
Тип промпта: structured

Ответ без RAG:
a spectrometer

Ответ с RAG:
Methods/Approach: [how the research was conducted, if mentioned]

--------------------------------------------------------------------------------

ТЕСТ 3/7
Вопрос: What is the TRAPPIST system?
Тип промпта: key_points

Ответ без RAG:
psychiatric system

Ответ с RAG:
Star TRAPPIST-1 hosts a system of seven transiting, terrestrial exoplanets

--------------------------------------------------------------------------------

ТЕСТ 4/7
Вопрос: What methods are used to detect exoplanets?
Тип промпта: scientific

Ответ без RAG:
spectrometers

Ответ с RAG:
Identify research methods and observational techniques mentioned

---------------

In [ ]:
print("=" * 80)
print("СРАВНЕНИЕ РАЗЛИЧНЫХ ТИПОВ ПРОМПТОВ")
print("=" * 80)

test_query = "What are habitable exoplanets and how are they detected?"

prompt_types = ["detailed", "key_points", "concise"]

print(f"\nВопрос: {test_query}\n")

for prompt_type in prompt_types:
    print(f"\n{'='*80}")
    print(f"Тип промпта: {prompt_type.upper()}")
    print('='*80)
    
    result = improved_rag_query(test_query, top_k=10, prompt_type=prompt_type, correct_spelling=False)
    
    print(f"Ответ:\n{result['answer']}")
    # print(f"\nИсточники: {len(result['sources'])}")
    # for i, source in enumerate(result['sources'][:2], 1):
    #     print(f"  {i}. {source['title'][:60]}... (сходство: {source['similarity']:.4f})")


СРАВНЕНИЕ РАЗЛИЧНЫХ ТИПОВ ПРОМПТОВ

Вопрос: What are habitable exoplanets and how are they detected?


Тип промпта: DETAILED
Ответ:
Theoretical studies exploring the formation and evolution of exomoons

Тип промпта: KEY_POINTS
Ответ:
Two decades ago, astronomers began detecting planets orbiting stars other than our Sun, so-called exoplanets.

Тип промпта: CONCISE
Ответ:
exomoons
